# 07 — Book Q&A Pairs

A model that can only write code but can't explain it is half-trained. This notebook
extracts Q&A pairs from every ARO book so the model learns to *reason about* ARO in
prose — essential for answering developer questions, explaining errors, and guiding users.

For each section in each chapter, the model generates questions a developer learning
ARO might ask. Answers are drawn directly from the section text, keeping them grounded
in the authoritative documentation rather than hallucinated.

**Books covered:**
- **TheLanguageGuide** — complete ARO reference (47 chapters + appendices)
- **AROByExample** — hands-on worked examples (14 chapters)
- **TheConstructionStudies** — advanced design patterns
- **TheInteractiveDialog** — conversational explanations
- **ThePluginGuide** — plugin development
- **TheShortStudies** — focused deep-dives
- **Reference** — action and grammar reference

**Input:**  `../../Book/` (all `.md` files)
**Output:** `../data/02_knowledge/knowledge_pairs.jsonl` (appended)

In [ ]:
import sys, importlib
from pathlib import Path
_cfg_dir = Path('.').resolve()
if _cfg_dir not in [Path(p) for p in sys.path]:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *
print(f'Config loaded | model={MODEL_ID}')
print(f'ARO root:  {ARO_ROOT}')
print(f'Pairs:     {PAIRS_FILE}')
print(f'Adapters:  {ADAPTER_DIR}')

import json, re, sys, random, subprocess
from pathlib import Path
from collections import Counter

# Load knowledge base for system prompt construction
with open(DATA_DIR / 'knowledge.json') as f:
    kb = json.load(f)

print(f'Book root: {BOOK_ROOT}')
print(f'Pairs file: {PAIRS_FILE}')
model, tokenizer, _, mlx_generate, make_sampler = load_model()


def _show_sample(pairs, n=2, label=''):
    import random as _r
    sample_pool = _r.sample(pairs, min(n, len(pairs)))
    print(f'\n── Sample ({label}, {len(pairs)} total) ──────────────────────')
    for i, s in enumerate(sample_pool, 1):
        if 'messages' in s:
            user = s['messages'][1]['content'] if len(s['messages']) > 1 else ''
            asst = s['messages'][2]['content'] if len(s['messages']) > 2 else ''
        else:
            user = s.get('instruction', s.get('user', ''))
            asst = s.get('output', s.get('assistant', ''))
        task = s.get('task_type', s.get('source', '?'))
        print(f'  [{i}] task={task}')
        print(f'       USER: {user[:120].strip()!r}')
        print(f'       ASST: {asst[:120].strip()!r}')
    print('─' * 60)

In [ ]:
# ── System prompt ────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert ARO language teacher creating training data.
ARO (Action Result Object) is a DSL for expressing business logic as natural-language statements.

SYNTAX:
  (FeatureSetName: BusinessActivity) {
      Verb [the] <result[:qualifier]> preposition [the] <object[:qualifier]>.
  }

KEY RULES:
- Articles (a/an/the) are optional everywhere
- String concatenation: ++ (NOT +)
- Variable names: hyphenated lowercase e.g. <user-id>, <order-total>
- Forbidden variable name prefixes: is-, with-, empty-
- Variables are IMMUTABLE — bind once; use a new name for each transformation
- Exactly ONE Application-Start per application (error if 0 or multiple)
- openapi.yaml REQUIRED for HTTP server; operationId must match feature set name
- Long-running apps: Keepalive the <application> for the <events>.

HTTP:
- Path params:   Extract the <id> from the <pathParameters: id>.
- Request body:  Extract the <data> from the <request: body>.
- Return statuses: <OK: status>, <Created: status>, <NoContent: status>,
                   <NotFound: status>, <BadRequest: status>, <Conflict: status>,
                   <Unauthorized: status>, <InternalServerError: status>

CONTROL FLOW:
- Conditional:   when <condition> { statements }
- Loop:          For each <item> in <list> { statements }
- Match:         match <var> { case value { statements } case other { statements } }
- Guard on declaration: (Name: EventName Handler) when <field> = value { ... }

COMPUTE & ARITHMETIC:
- Compute the <total> from <price> * <qty>.
- Compute the <upper: uppercase> from <text>.
- Compute the <len: length> from <text>.
- Supported ops: +, -, *, /, % (integers); ++ (strings)

CROSS-FEATURE SHARING:
- Publish as <alias> <variable>.  (makes variable accessible across feature sets in same business activity)

EVENTS:
- Emit a <EventName: event> with <data>.
- Handler:  (HandlerName: EventName Handler) { ... }
- State:    Accept the <new-state: toState> for the <entity>.

LIFECYCLE:
- Application-Start: required entry point
- Application-End: Success (optional graceful shutdown)
- Application-End: Error (optional crash handler)

AVAILABLE ACTIONS (verb → role → prepositions):
- Extract     REQUEST   from/with     — pull data from request/event/object
- Retrieve    REQUEST   from/where    — fetch from repository or service
- Fetch       REQUEST   from/with     — HTTP GET external URL
- Parse       REQUEST   from          — parse JSON/YAML/HTML/CSV text
- Request     REQUEST   from/with     — HTTP POST/PUT/PATCH
- Compute     OWN       from/with/for — arithmetic, string ops, built-in transforms
- Transform   OWN       from/with     — type coercion (e.g. string → int)
- Validate    OWN       for/with      — check constraints, return bool
- Compare     OWN       against/with  — compare two values, return bool
- Create      OWN       with          — instantiate struct or entity
- Merge       OWN       with          — combine objects or collections
- Filter      OWN       where/with    — select subset of collection
- Sort        OWN       by/with       — order a collection
- Split       OWN       by/from       — split string or list by delimiter
- Join        OWN       with/from     — join strings or list elements
- Accept      OWN       with/for      — state machine transition
- Return      RESPONSE  with/for      — send HTTP response and end feature set
- Throw       RESPONSE  with/for      — raise exception (propagates as error)
- Store       EXPORT    into          — persist to repository (auto-generates id)
- Update      EXPORT    into          — update existing record in repository
- Delete      EXPORT    from          — remove record from repository
- Log         EXPORT    to            — write to console or log service
- Send        EXPORT    to/with       — send email or message
- Emit        EXPORT    with          — publish domain event to EventBus
- Publish     EXPORT    as            — share variable across feature sets
- Start       EXPORT    with          — start a service (HTTP server, file monitor)
- Stop        EXPORT    with          — stop a running service
- Keepalive   EXPORT    for           — block until shutdown signal (for servers)
- Render      EXPORT    with/using    — render Mustache template
- Notify      EXPORT    with          — send notification to user(s)
- Configure   EXPORT    with          — set runtime configuration option
- Read        REQUEST   from          — read file contents
- Write       EXPORT    to            — write data to file
- Copy        EXPORT    to            — copy file to destination
- Move        EXPORT    to            — move file to destination

Always wrap ARO code in ```aro ... ``` fences.
Always generate complete, valid ARO that would pass `aro check`.
"""


def chat(messages, max_tokens=800):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return mlx_generate(
        model, tokenizer, prompt=prompt,
        max_tokens=max_tokens, verbose=False,
        sampler=make_sampler(temp=0.5),   # slightly higher temp for question diversity
    )


# ── Resume support ───────────────────────────────────────────────────────────
from collections import Counter as _Counter
_done_keys = set()
_pair_count = 0
if PAIRS_FILE.exists():
    with open(PAIRS_FILE) as f:
        for line in f:
            if line.strip():
                try:
                    s = json.loads(line)
                    _done_keys.add((s.get('source',''), s.get('instruction','')[:80]))
                    _pair_count += 1
                except Exception:
                    pass
# Build set of section sources that already have ≥3 pairs (fully done)
_src_counts = _Counter(src for src, _ in _done_keys)
_done_sources = {src for src, n in _src_counts.items() if n >= 3}
print(f'Resuming — {_pair_count} pairs already in knowledge_pairs.jsonl, '
          f'{len(_done_sources)} sections fully done')


def save_pair(source, instruction, output, score=1.0, metadata=None):
    key = (source, instruction[:80])
    if key in _done_keys:
        return False
    with open(PAIRS_FILE, 'a') as f:
        f.write(json.dumps({
            'instruction': instruction,
            'output':      output,
            'source':      source,
            'score':       score,
            'metadata':    metadata or {},
        }) + '\n')
    _done_keys.add(key)
    return True


print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
print('Helpers ready.')

## Discover Book Chapters

Collect every `.md` file from every book collection.

In [ ]:
# Each entry: {book, chapter_name, path, content}
all_chapters = []

for book_dir in sorted(BOOK_ROOT.iterdir()):
    if not book_dir.is_dir():
        continue
    book_name = book_dir.name
    md_files  = sorted(book_dir.glob('*.md'))
    for md_path in md_files:
        content = md_path.read_text().strip()
        if len(content) < 200:   # skip near-empty stubs
            continue
        all_chapters.append({
            'book':    book_name,
            'name':    md_path.stem,
            'path':    md_path,
            'content': content,
        })
    print(f'  {book_name}: {len(md_files)} files')

print(f'\nTotal chapters to process: {len(all_chapters)}')

## Q&A Extraction Loop

For each chapter:
1. Split into sections by heading (`##`, `###`)
2. For each substantive section, ask the LLM to generate 2–3 questions a developer might ask
3. The answer is drawn directly from the section text (+ optional LLM polish)
4. Save each (question, answer) pair to `knowledge_pairs.jsonl`

This teaches the model *how to explain* ARO concepts in prose — complementing
the code-generation pairs from notebooks 02 and 03.

In [ ]:
def split_into_sections(content):
    """
    Split a markdown document into (heading, body) pairs.
    Top-level chapter prose (before any heading) is included as heading=chapter_title.
    """
    # Extract chapter title from first # heading
    title_match = re.search(r'^#\s+(.+)', content, re.MULTILINE)
    chapter_title = title_match.group(1).strip() if title_match else 'Introduction'

    # Split on ## or ### headings
    parts = re.split(r'(?m)^(#{2,3}\s+.+)$', content)

    sections = []
    # First part: content before first ## heading
    if parts[0].strip() and len(parts[0].strip()) > 100:
        sections.append((chapter_title, parts[0].strip()))

    # Subsequent heading+body pairs
    i = 1
    while i < len(parts) - 1:
        heading = re.sub(r'^#{2,3}\s+', '', parts[i]).strip()
        body    = parts[i + 1].strip() if i + 1 < len(parts) else ''
        if body and len(body) > 120:   # skip very short sections
            sections.append((heading, body))
        i += 2

    return sections


# Matches questions that reference code/examples without including any
_VAGUE_REF_RE = re.compile(
    r'\b(this example|this code|the following|shown above|this snippet|'
    r'the above|this feature set|this statement|this application)\b',
    re.IGNORECASE,
)


def generate_qa_pairs(book, chapter_name, heading, section_body):
    """
    Ask the LLM to generate Q&A pairs from a book section.
    Returns list of (question, answer) tuples.

    When the section contains code examples, the prompt instructs the model
    to embed a relevant snippet directly in the question so questions like
    "How does this work?" are self-contained and unambiguous as training data.
    """
    body_trimmed = section_body[:3000]

    # Extract code blocks from the section for grounding
    code_blocks = re.findall(
        r'```(?:aro|swift|yaml|json|bash|sh)?\n(.*?)```',
        body_trimmed, re.DOTALL,
    )

    code_hint = ''
    if code_blocks:
        snippets = [f'```aro\n{cb.strip()[:300]}\n```' for cb in code_blocks[:2]]
        code_hint = (
            '\n\nThe section contains these code examples. '
            'When a question refers to code or an example, include the relevant '
            'snippet directly in the question (e.g. "What does this do?\\n```...```):\n'
            + '\n'.join(snippets)
        )

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': (
            f'You are creating Q&A training pairs from the ARO language documentation.\n\n'
            f'Book: {book}\n'
            f'Chapter: {chapter_name}\n'
            f'Section: {heading}\n\n'
            f'--- Section Content ---\n{body_trimmed}\n--- End ---'
            f'{code_hint}\n\n'
            f'Generate 3 questions a developer learning ARO might ask after reading '
            f'this section.\n'
            f'For each question, write a clear, accurate answer GROUNDED SOLELY in '
            f'the section content above.\n'
            f'- Do NOT invent features, actions, or syntax not shown in the section\n'
            f'- Include the actual ARO code from the section if the answer involves code\n'
            f'- If your question references "this code" or "this example", embed the '
            f'code inline in the question\n\n'
            f'Format EXACTLY like this:\n\n'
            f'### Q1\n'
            f'**Question:** <specific, self-contained question>\n'
            f'**Answer:** <answer grounded in the section above>\n\n'
            f'### Q2\n'
            f'**Question:** ...\n'
            f'**Answer:** ...\n\n'
            f'### Q3\n'
            f'**Question:** ...\n'
            f'**Answer:** ...'
        )},
    ]

    output = chat(messages, max_tokens=900)

    # Parse Q&A blocks
    pairs = []
    blocks = re.split(r'###\s*Q\d+', output)
    for block in blocks[1:]:
        q_match = re.search(
            r'\*\*Question:\*\*\s*(.+?)(?=\n\*\*Answer|\Z)', block, re.DOTALL
        )
        a_match = re.search(
            r'\*\*Answer:\*\*\s*(.+?)(?=\n###|\Z)', block, re.DOTALL
        )
        if q_match and a_match:
            question = q_match.group(1).strip().replace('\n', ' ')
            answer   = a_match.group(1).strip()
            if len(question) > 10 and len(answer) > 20:
                pairs.append((question, answer))

    # Post-process: inject first code block into questions that reference
    # "this example" / "this code" etc. but contain no inline code.
    # This makes training pairs self-contained even when the model forgets.
    if code_blocks:
        first_code = code_blocks[0].strip()[:400]
        grounded = []
        for q, a in pairs:
            if _VAGUE_REF_RE.search(q) and '```' not in q and '`' not in q:
                q = q + f'\n\n```aro\n{first_code}\n```'
            grounded.append((q, a))
        pairs = grounded

    return pairs


# ── Main loop ────────────────────────────────────────────────────────────────
print(f'\n--- Q&A Extraction: {len(all_chapters)} chapters ---')
total_new = 0
collected_pairs = []

for ch in all_chapters:
    book         = ch['book']
    chapter_name = ch['name']
    content      = ch['content']
    source_prefix = f'book_qa:{book}:{chapter_name}'

    if not re.search(r'```aro', content, re.IGNORECASE):
        continue  # skip chapters with no ARO code blocks

    sections = split_into_sections(content)
    chapter_new = 0

    for heading, body in sections:
        source_key = f'{source_prefix}:{heading[:50]}'

        # Skip if this section already has ≥3 pairs (resume support)
        if source_key in _done_sources:
            continue

        pairs = generate_qa_pairs(book, chapter_name, heading, body)
        for question, answer in pairs:
            if save_pair(
                source_key,
                question,
                answer,
                score=1.0,
                metadata={'book': book, 'chapter': chapter_name, 'section': heading},
            ):
                chapter_new += 1
                total_new   += 1
                collected_pairs.append({
                    'instruction': question,
                    'output': answer,
                    'source': source_key,
                    'task_type': 'book_qa',
                })

    if chapter_new > 0:
        print(f'  {book}/{chapter_name}: +{chapter_new} pairs', flush=True)

print(f'\nDone: {total_new} new Q&A pairs added')
_show_sample(collected_pairs, label='NB07 book_qa')

## Summary

In [ ]:
all_pairs = []
with open(PAIRS_FILE) as f:
    for line in f:
        if line.strip():
            try:
                all_pairs.append(json.loads(line))
            except Exception:
                pass

sources = Counter(p.get('source', '').split(':')[0] for p in all_pairs)

print(f'Total pairs in knowledge_pairs.jsonl: {len(all_pairs)}')
print('\nBy source type:')
for src, n in sorted(sources.items(), key=lambda x: -x[1]):
    print(f'  {src:30s}: {n}')

qa_pairs = [p for p in all_pairs if p.get('source', '').startswith('book_qa:')]
books    = Counter(p.get('source', '').split(':')[1] for p in qa_pairs)
print(f'\nQ&A pairs by book ({len(qa_pairs)} total):')
for book, n in sorted(books.items(), key=lambda x: -x[1]):
    print(f'  {book:35s}: {n}')

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import date as _date

_run_dir = Path('.') / 'run' / _date.today().isoformat()
_run_dir.mkdir(parents=True, exist_ok=True)
_out = _run_dir / '07_book_qa_pairs.png'

_qa_pairs = [p for p in all_pairs if p.get('source', '').startswith('book_qa:')]
_book_counts = {}
for p in _qa_pairs:
    book = p.get('source', '').split(':')[1] if ':' in p.get('source', '') else 'unknown'
    _book_counts[book] = _book_counts.get(book, 0) + 1

_sorted = sorted(_book_counts.items(), key=lambda x: x[1])
_labels = [b.replace('The', 'The\n') for b, _ in _sorted]
_values = [n for _, n in _sorted]
_colors = plt.cm.Blues([0.4 + 0.6 * i / max(1, len(_values) - 1) for i in range(len(_values))])

fig, ax = plt.subplots(figsize=(10, max(4, len(_labels) * 0.65)))
bars = ax.barh(_labels, _values, color=_colors, edgecolor='white', height=0.65)
ax.set_xlabel('Q&A pairs generated')
ax.set_title(
    f'Book Q&A Pairs — {len(_qa_pairs):,} total across {len(_book_counts)} books',
    fontsize=13, fontweight='bold'
)
for bar, n in zip(bars, _values):
    ax.text(n + max(_values) * 0.01, bar.get_y() + bar.get_height() / 2,
            str(n), va='center', fontsize=9, color='#444')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
ax.set_facecolor('#f9f9f9')
fig.patch.set_facecolor('#fafafa')
fig.tight_layout()
fig.savefig(_out, dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {_out}')


In [ ]:
# ── Samples per category (book) ───────────────────────────────────────────────
import json, random
from collections import defaultdict

_qa_pairs = [p for p in all_pairs if p.get('source', '').startswith('book_qa:')]

_by_book = defaultdict(list)
for p in _qa_pairs:
    book = p.get('source', '').split(':')[1] if ':' in p.get('source', '') else 'unknown'
    _by_book[book].append(p)

SAMPLES_PER_BOOK = 2

for book in sorted(_by_book):
    pool = _by_book[book]
    print(f'\n{"─"*72}')
    print(f'  {book}  ({len(pool)} pairs)')
    print('─'*72)
    for s in random.sample(pool, min(SAMPLES_PER_BOOK, len(pool))):
        chapter = s['source'].split(':')[2] if s['source'].count(':') >= 2 else ''
        print(f'[{chapter}]')
        print(f'Q: {s["instruction"][:300]}')
        out = s.get('output', '')
        print(f'A: {out[:320]}{"..." if len(out) > 320 else ""}')
        print()